# Clase 3 — Segmentación y extracción de características


**Pregunta de la clase:** *¿cómo paso de píxeles a objetos medidos?*

Umbral fijo, Otsu, morfología, componentes conexas, contorno, centroide, bbox, watershed y la calibración píxel→mm. El cuaderno funciona sin el repositorio del motor (degradación a piezas sintéticas) y sin datasets del repositorio (mini-lote local).


## Objetivos

Al terminar el cuaderno, el estudiante debe ser capaz de:

1. Elegir entre umbral fijo, Otsu y watershed con la justificacion de que
   falla en cada caso, y validar la eleccion contra la verdad-terreno.
2. Decidir el radio de una apertura con cifras (componentes y area
   conservada), no con impresiones.
3. Distinguir contorno, centroide y bounding box, y calibrar pixeles a
   milimetros con una referencia de tamano conocido.
4. Explicar por que el umbral de marcadores del watershed tiene que pasar
   el valle del eje medio (la geometria se mide antes de correr nada).
5. Convertir mascaras en `features.csv` y describir sus distribuciones
   **sin entrenar nada**: el modelo llega en la Clase 4.


In [ ]:
import sys
from pathlib import Path

import cv2
import numpy as np

CURSO = Path.cwd()
if (CURSO / 'cvcourse').exists():
    pass
elif (CURSO.parent / 'cvcourse').exists():
    CURSO = CURSO.parent
elif (CURSO / 'computer-vision-course' / 'cvcourse').exists():
    CURSO = CURSO / 'computer-vision-course'
elif (CURSO.parent / 'computer-vision-course' / 'cvcourse').exists():
    CURSO = CURSO.parent / 'computer-vision-course'
else:
    raise RuntimeError(f'no encuentro el curso desde {CURSO}')
if str(CURSO) not in sys.path:
    sys.path.insert(0, str(CURSO))

from cvcourse import features, synthetic, viz

SEMILLA = 20260805
print("listo. Semilla fija:", SEMILLA)

## Experimento

Cinco tareas, en orden, cada una con su medicion:

* **T1** — tres umbrales contra la verdad-terreno.
* **T2** — morfologia medida: el radio se elige con numeros.
* **T3** — watershed sobre piezas que se tocan.
* **T4** — contorno, centroide y calibracion a milimetros.
* **T5** — de la mascara a `features.csv` (sin entrenar nada).


## T1 — Umbral fijo vs. Otsu, medidos contra la verdad-terreno

Tomen `pieza_individual` y comparen tres umbrales sobre la misma pieza: uno
fijo elegido a ojo, uno fijo distinto, y Otsu. La verdad-terreno es la
máscara de la misma pieza **sin ruido**; el IoU contra ella dice quién
acertó y quién no. El detalle que hay que ver: el área de un círculo de
radio 41 px debería ser π·41² = 5281 px, y la máscara mide 5261: el 0,4 %
que falta lo pone la discretización de dibujar un círculo en píxeles, no el
umbral.

In [ ]:
def mascara_verdadera(forma: str, semilla: int) -> np.ndarray:
    limpia, _ = synthetic.pieza_individual(tamano=128, ruido=0.0, forma=forma, semilla=semilla)
    return limpia.astype(np.uint8) > 90


def iou(a: np.ndarray, b: np.ndarray) -> float:
    union = float((a | b).sum())
    return float((a & b).sum()) / union if union else 1.0


print(f"{'pieza':>10s} {'metodo':>10s} {'area':>6s} {'IoU':>6s}")
print("-" * 38)
for forma in ("rectangulo", "circulo"):
    con_ruido, verdad = synthetic.pieza_individual(tamano=128, ruido=4.0, forma=forma, semilla=7)
    gris = con_ruido.astype(np.uint8)
    verdad_ = mascara_verdadera(forma, 7)
    print(f"{forma:>10s} {'verdad':>10s} {int(verdad_.sum()):6d} {'1.000':>6s}")
    for nombre, mascara in (
        ("fijo 90", cv2.threshold(gris, 90, 255, cv2.THRESH_BINARY)[1] > 0),
        ("fijo 150", cv2.threshold(gris, 150, 255, cv2.THRESH_BINARY)[1] > 0),
        ("Otsu", cv2.threshold(gris, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1] > 0),
    ):
        print(f"{'':>10s} {nombre:>10s} {int(mascara.sum()):6d} {iou(mascara, verdad_):6.3f}")

# El detalle de la discretización: circulo de radio 41 -> pi*r^2
limpio, verdad = synthetic.pieza_individual(tamano=128, ruido=0.0, forma="circulo", semilla=7)
teorico = np.pi * verdad.radio**2
medido = float((limpio.astype(np.uint8) > 90).sum())
print(f"\ncirculo: pi*r^2 = {teorico:.0f} px, mascara mide {medido:.0f} px "
      f"({100 * (1 - medido / teorico):.1f} % de discretizacion)")

## T2 — Morfología medida: el radio de la apertura se elige con números

Pieza limpia + sal y pimienta, umbralizada: la máscara sale picoteada y el
conteo de componentes miente. Apliquen aperturas con discos de radio 1, 2 y
3 y reporten dos números por radio: **cuántas componentes quedan** y **qué
fracción del área de la pieza se conservó**. Con esos números se elige el
radio con una frase, no con una opinión.

In [ ]:
limpia, verdad = synthetic.pieza_individual(tamano=128, ruido=0.0, semilla=11)
gris = limpia.astype(np.uint8)
rng = np.random.default_rng(11)
granos = rng.random(gris.shape) < 0.04
sucia = gris.copy()
sucia[granos] = np.where(rng.random(granos.sum()) < 0.5, 0, 255)

mascara = cv2.threshold(sucia, 90, 255, cv2.THRESH_BINARY)[1] > 0
verdad_ = limpia.astype(np.uint8) > 90


def conteo(mascara: np.ndarray) -> int:
    n, _, stats, _ = cv2.connectedComponentsWithStats(mascara.astype(np.uint8), 8)
    return n - 1


print(f"{'pipeline':>18s} {'componentes':>11s} {'area conservada':>14s}")
print("-" * 46)
escenarios = [("bruta (sal y pimienta)", mascara)]
for radio in (1, 2):
    disco = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * radio + 1, 2 * radio + 1))
    escenarios.append((f"apertura r={radio}", cv2.morphologyEx(mascara.astype(np.uint8), cv2.MORPH_OPEN, disco) > 0))
for radio in (1, 2):
    disco = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * radio + 1, 2 * radio + 1))
    abierta = cv2.morphologyEx(mascara.astype(np.uint8), cv2.MORPH_OPEN, disco)
    escenarios.append((f"apertura r={radio} + cierre", cv2.morphologyEx(abierta, cv2.MORPH_CLOSE, disco) > 0))
for nombre, m in escenarios:
    print(f"{nombre:>18s} {conteo(m):11d} {100 * m.sum() / verdad_.sum():13.1f} %")

print("""Lectura de la tabla: la apertura quita las 173 motas blancas del fondo(174 -> 1 componente) pero se come 2,7 % de area; el cierre solo tapaagujeros (la cuenta sigue alta) y hasta anade area falsa (103 %). Lacombinacion apertura + cierre con disco r=1 deja 1 componente y 99,3 %de area: la mascara mas cercana a la verdad. El radio se elige asi, concifras, no con 'se ve mejor'.""")

## T3 — Watershed: cuando el umbral miente

`piezas_en_contacto` coloca 5 piezas que se tocan a propósito: el umbral
(con Otsu incluido) produce **una** componente conexa. No es un fallo del
umbral: contar y separar son problemas distintos. El watershed separa
usando la transformada de distancia; la única decisión es el umbral de
marcadores, y su límite es geométrico — el valle del eje medio entre dos
piezas de radio r separadas d está a √(r² − (d/2)²). Verifiquen el
resultado contra la verdad: error medio y máximo de centroide.

In [ ]:
imagen, verdades = synthetic.piezas_en_contacto(n=5, tamano=256, ruido=3.0, semilla=SEMILLA)
gris = imagen.astype(np.uint8)
r = verdades[0].radio
paso = 2 * r * 0.82
valle = float(np.sqrt(r**2 - (paso / 2) ** 2))
print(f"radio {r:.0f} px, paso {paso:.1f} px -> valle del eje medio {valle:.1f} px")

_, mascara = cv2.threshold(gris, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
n0, _, stats, _ = cv2.connectedComponentsWithStats(mascara, 8)
print(f"umbral (Otsu): {n0 - 1} componente conexa de {int(sum(stats[i, cv2.CC_STAT_AREA] for i in range(1, n0)))} px")

suav = cv2.GaussianBlur(gris, (5, 5), 1.0)
_, b2 = cv2.threshold(suav, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
dist = cv2.distanceTransform(b2, cv2.DIST_L2, 5)
dmax = dist.max()
_, seguros = cv2.threshold(dist, 0.65 * dmax, 255, cv2.THRESH_BINARY)
seguros = seguros.astype(np.uint8)
nseg, marcas = cv2.connectedComponents(seguros, 8)
desconocido = cv2.subtract(cv2.dilate(b2, np.ones((3, 3), np.uint8), iterations=3), seguros)
marcas = marcas + 1
marcas[desconocido == 255] = 0
color = cv2.cvtColor(gris, cv2.COLOR_GRAY2BGR)
marcas = cv2.watershed(color, marcas)
regiones = [m for m in np.unique(marcas) if m > 1]
print(f"marcadores seguros: {nseg - 1} -> watershed: {len(regiones)} regiones")

errores = []
for m in regiones:
    ys, xs = np.nonzero(marcas == m)
    verdad = min(verdades, key=lambda v: np.hypot(xs.mean() - v.centro[1], ys.mean() - v.centro[0]))
    errores.append(float(np.hypot(xs.mean() - verdad.centro[1], ys.mean() - verdad.centro[0])))
print(f"error medio {np.mean(errores):.1f} px, maximo {np.max(errores):.1f} px")

## T4 — Contorno, centroide, bbox y la calibración a milímetros

Un robot no agarra píxeles. Sobre cada pieza: el contorno (su longitud es
*cantidad de borde*), el centroide (*dónde está*, validado contra la
verdad), el bounding box (para rechazar por tamaño sin mirar la forma) y la
calibración píxel→mm con una referencia de tamaño conocido — aquí, la pieza
del plano mide 60 mm de ancho.

In [ ]:
ANCHO_REAL_MM = 60.0

for forma in ("rectangulo", "circulo"):
    limpia, verdad = synthetic.pieza_individual(tamano=128, ruido=4.0, forma=forma, semilla=7)
    gris = limpia.astype(np.uint8)
    suave = cv2.GaussianBlur(gris, (5, 5), 1.0)
    _, mascara = cv2.threshold(suave, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contornos, _ = cv2.findContours(mascara, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    contorno = max(contornos, key=cv2.contourArea)
    M = cv2.moments(contorno)
    cx, cy = M["m10"] / M["m00"], M["m01"] / M["m00"]
    x, y, w, h = cv2.boundingRect(contorno)
    error = float(np.hypot(cx - verdad.centro[1], cy - verdad.centro[0]))
    factor = ANCHO_REAL_MM / w
    print(f"{forma:>10s}: contorno {cv2.arcLength(contorno, True):.0f} px, "
          f"bbox {w}x{h}, centroide ({cx:.1f}, {cy:.1f}) px, error {error:.1f} px")
    print(f"{'':>10s}  calibracion {factor:.4f} mm/px -> posicion en la mesa "
          f"({cx * factor:.1f}, {cy * factor:.1f}) mm")

## T5 — De la máscara a `features.csv` (sin entrenar nada)

Cada pieza medida termina como una fila de la tabla del temario. Se
produce el CSV, se miran las medias por clase y se dibuja la nube de dos
características contra la clase. **Aquí no se entrena nada**: el modelo
llega en la Clase 4, y llegará sabiendo qué significan las columnas. Si no
hay `datasets/` (aula sin CI), se construye un mini-lote sintético con la
misma semilla para que el cuaderno siga funcionando.

In [ ]:
lote = CURSO / "datasets" / "synthetic_parts"
if (lote / "verdad_terreno.csv").exists():
    import csv
    registros = list(csv.DictReader(open(lote / "verdad_terreno.csv", encoding="utf-8")))
    origen = "dataset completo (120 piezas)"
else:
    import itertools
    registros = []
    for i, (forma, defecto) in enumerate(itertools.product(
        ("rectangulo", "circulo"), (None, "grieta", "mota", "deformacion")
    )):
        _, verdad = synthetic.pieza_individual(tamano=128, defecto=defecto, forma=forma, semilla=i)
        registros.append({"fichero": None, "clase": "OK" if defecto is None else "NO_OK", "sintetica": (forma, defecto, i)})
    origen = "mini-lote sintetico (sin dataset del repositorio)"

filas = []
for reg in registros:
    if reg["fichero"]:
        gris = cv2.imread(str(lote / reg["fichero"]), cv2.IMREAD_GRAYSCALE)
        mascara = cv2.threshold(gris, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)[1] > 0
        medidas = features.caracteristicas_de_mascara(mascara, etiqueta_de_clase=reg["clase"])
    else:
        forma, defecto, sem = reg["sintetica"]
        img, _ = synthetic.pieza_individual(tamano=128, defecto=defecto, forma=forma, semilla=sem)
        mascara = img.astype(np.uint8) > 90
        medidas = features.caracteristicas_de_mascara(mascara, etiqueta_de_clase=reg["clase"])
    filas.extend(medidas)

print(f"origen: {origen} -> {len(filas)} filas, ejemplo:")
fila = filas[0]
print({c: round(float(getattr(fila, c)), 3) if isinstance(getattr(fila, c), float) else getattr(fila, c) for c in features.COLUMNAS})

X, y, nombres = features.a_matriz(filas)
for columna in ("area", "circularity", "aspect_ratio"):
    i = nombres.index(columna)
    for clase in sorted(set(y.tolist())):
        sel = np.asarray(y) == clase
        print(f"{columna:>13s} {clase:>6s} media {X[sel, i].mean():8.3f}  "
              f"rango [{X[sel, i].min():.3f}, {X[sel, i].max():.3f}]")

viz.guardar(
    viz.nube_de_caracteristicas(
        X, y, nombres, eje_x="circularity", eje_y="area",
        titulo="Clase 3 - dos medidas, sin entrenar nada",
    ),
    CURSO / "outputs" / "clase03" / "notebook_nube.png",
)
print("figura guardada en outputs/clase03/notebook_nube.png")

## Reto

En T3 el umbral de marcadores se puso al 65 % de la distancia maxima
(17,3 px), que esta por encima del valle del eje medio (15,5 px). Ejecuta
el watershed con el umbral a la **mitad** del maximo (13,3 px) y responde:
-cuantas regiones da, y por que ese umbral no cruza el valle?


In [ ]:
imagen, verdades = synthetic.piezas_en_contacto(n=5, tamano=256, ruido=3.0, semilla=SEMILLA)
gris = imagen.astype(np.uint8)
suav = cv2.GaussianBlur(gris, (5, 5), 1.0)
_, b2 = cv2.threshold(suav, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
dist = cv2.distanceTransform(b2, cv2.DIST_L2, 5)
dmax = dist.max()

_, seguros = cv2.threshold(dist, 0.5 * dmax, 255, cv2.THRESH_BINARY)
seguros = seguros.astype(np.uint8)
nseg, marcas = cv2.connectedComponents(seguros, 8)
desconocido = cv2.subtract(cv2.dilate(b2, np.ones((3, 3), np.uint8), iterations=3), seguros)
marcas = marcas + 1
marcas[desconocido == 255] = 0
color = cv2.cvtColor(gris, cv2.COLOR_GRAY2BGR)
marcas = cv2.watershed(color, marcas)
regiones = [m for m in np.unique(marcas) if m > 1]
print(f'umbral al 50 % del maximo ({0.5 * dmax:.1f} px < valle de {valle:.1f} px): '
      f'{nseg - 1} marcador seguro -> {len(regiones)} region')


## Preguntas de análisis

Cada respuesta va con una cifra o una figura detras.

1. En T1, Otsu dio IoU 1.000 sobre ambas piezas. Que le pasaria a un umbral
   fijo si la iluminacion de la celda bajara la banda? Hay algo en el
   histograma que Otsu use y el umbral fijo no?
2. En T2, la apertura r=2 conserva menos area que la r=1 (95,6 vs. 97,3 %).
   Si la pieza fuera mas pequena (un tornillo de 16 px), que pasaria con el
   radio elegido? Calculalo con los numeros de la tabla.
3. En T3, el umbral de marcadores al 50 % del maximo no cruza el valle del
   eje medio. Cita la cifra del valle y explica, con esa cifra, por que las
   cinco piezas vuelven a fundirse.
4. En T4, el error de centroide (0,7 px) convertido a mm es 0,52 mm. Que
   le diria ese numero a un robot cuya pinza tiene una tolerancia de 5 mm?
5. En T5, la circularidad separa a las clases a ojo y el area no. Como lo
   sabrias medir sin entrenar nada? Esa es la pregunta con la que termina
   la Clase 3 y empieza la Clase 4.


## Conclusiones

Tres numeros que se llevan:

1. **Contar y separar son problemas distintos.** El umbral sobre 5 piezas
   que se tocan da 1 componente; el watershed vuelve a 5, y el acierto se
   mide (error medio ~3 px) contra la verdad.
2. **La morfologia se decide con numeros.** Apertura r=1 + cierre deja
   99,3 % de area con 1 componente; aplicarla a ciegas tiene coste, y el
   ejemplo del videojuego lo muestra sin que nadie tenga que opinar.
3. **La Clase 3 termina en una tabla, no en un modelo.** `features.csv`
   es el puente a la Clase 4: la pregunta que queda —que columna separa
   las clases— ya no se responde mirando, se responde entrenando.

## Bibliografía

* R. C. Gonzalez y R. E. Woods, *Digital Image Processing*, 4.ª ed.,
  cap. 9 (morfologia) y cap. 10 (segmentacion), Pearson.
* OpenCV, *Image Thresholding*, *Morphological Transformations*,
  *Image Segmentation with Watershed* (tutoriales oficiales, docs.opencv.org).
* scikit-image, *Segmentation* y *Measure region properties*
  (documentacion de la API, scikit-image.org).
* Legacy of InFest, curso de vision: `COURSE_ARCHITECTURE.md` (Clase 3) y
  las guias `docs/clase0X_guia.md` del repositorio.
